In [0]:

from pyspark.sql.types import ArrayType, StructType, StructField, StringType
from pyspark.sql.functions import from_json, col, explode_outer

#################################
#######---order_products---######
#################################


def df_ordred_products(df):
    order_product_schema = ArrayType(
        StructType([
            StructField("curr", StringType()),
            StructField("id", StringType()),
            StructField("name", StringType()),
            StructField("price", StringType()),
            StructField("promotion_info", StringType()),
            StructField("qty", StringType()),
            StructField("unit", StringType())
            ])
        )


    df_ordred_products = df.withColumn("ordered_products", from_json(col("ordered_products"), order_product_schema))\
                .withColumn("ordered_products", explode_outer("ordered_products"))\
                .select(
                            "customer_id",
                            "customer_name",
                            "order_number",
                            "ordered_products.id",
                            col("ordered_products.name"). alias ("product_name"),
                            "ordered_products.price",
                            "ordered_products.qty",
                            "ordered_products.unit",
                            "ordered_products.curr"
                        )
    return df_ordred_products



#################################
#######---promotion---######
#################################


def df_promo(df):
    promo_info_schema = ArrayType(
        StructType([
            StructField("promo_disc", StringType()),
            StructField("promo_id", StringType()),
            StructField("promo_item", StringType()),
            StructField("promo_qty", StringType())
        ])
    )


    df_promo = df.withColumn("promo_info", from_json(col("promo_info"), promo_info_schema))\
                  .withColumn("promo_info", explode_outer("promo_info")).filter(col("promo_info").isNotNull())\
                  .select(
                            "customer_id",
                            "customer_name",
                            "order_number",
                            "promo_info.promo_id",
                            "promo_info.promo_item",
                            col("promo_info.promo_qty").alias("promo_quantity"),
                            col("promo_info.promo_disc").alias("promo_discount")
                         )
    return df_promo            



#################################
#######---maincode---######
#################################

print(f"Reading source table")
df = spark.read.table("ecommerce_analytics.bronze.sales_orders")

print(f"Transforming data - Processing Ordered Products Table")
df_ordred_products = df_ordred_products(df)
print(f"Writing data - Writing Ordered Products Table")
df_ordred_products.write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.order_products")

print(f"Transforming data - Processing promotions Table")
df_promo = df_promo(df)
print(f"Writing data - Writing Ordered Promotions Table")
df_promo.write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.promo_info")

print(F"Successfully wrote all tables") 